# Phase 10 — Business Impact
### Energy Demand Forecasting Project

**Question this phase answers:** Phases 1-9 established, rigorously, that
LightGBM is the strongest day-ahead forecaster on this dataset (with LSTM
statistically tied, and both well clear of Prophet/SARIMA/naive). This
phase asks the question a stakeholder actually cares about: *so what?*
What does a 19% MAE reduction over the naive baseline actually mean in
operational and financial terms, which model should be put into
production, and why?

**Scope honesty upfront:** this dataset is a single household's minute-
level consumption, not a utility's aggregate load. Two things follow from
that, both handled explicitly below rather than glossed over:
- The *relative* accuracy comparison between models (Phases 6-9) is real
  and directly usable as-is.
- Any *dollar* estimate requires scaling this single household's numbers
  up to a utility-scale portfolio, which needs explicit assumptions
  (portfolio size, imbalance pricing) this project cannot validate on its
  own. Every such number below is computed from a clearly labeled
  assumption cell, not asserted -- change the assumptions, get a different
  number, exactly as it should work.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import duckdb
import pandas as pd

from src.evaluation.splits import get_train_test_split
from src.evaluation.baselines import compute_metrics
from src.evaluation.results_store import load_results

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## The accuracy improvement, recomputed live (not hardcoded)

Reloading Phase 9's saved results rather than retyping its numbers -- if
any upstream result ever changes, this recomputes rather than silently
going stale.

In [3]:
baselines = load_results("baselines")
sarima_arima = load_results("sarima_arima")
prophet_results = load_results("prophet")
lightgbm_results = load_results("lightgbm_direct")
lstm_results = load_results("lstm")

all_24h = pd.concat(
    [baselines, sarima_arima, prophet_results, lightgbm_results, lstm_results], ignore_index=True,
)
CANONICAL = ["daily_seasonal_naive", "sarima", "prophet", "lightgbm", "lstm"]
summary_24h = pd.DataFrame({
    m: compute_metrics(all_24h[all_24h["method"] == m]) for m in CANONICAL
}).T.sort_values("MAE")

baseline_mae, baseline_rmse = summary_24h.loc["daily_seasonal_naive", ["MAE", "RMSE"]]
lgbm_mae, lgbm_rmse = summary_24h.loc["lightgbm", ["MAE", "RMSE"]]

mae_reduction_kw = baseline_mae - lgbm_mae
mae_reduction_pct = mae_reduction_kw / baseline_mae * 100
rmse_reduction_pct = (baseline_rmse - lgbm_rmse) / baseline_rmse * 100

print(summary_24h)
print()
print(f"LightGBM vs. baseline: MAE down {mae_reduction_kw:.3f} kW ({mae_reduction_pct:.1f}%), "
      f"RMSE down {rmse_reduction_pct:.1f}%")

                        MAE   RMSE    MAPE   sMAPE          n
lightgbm             0.4287 0.5913 61.9708 44.0040 6,156.0000
lstm                 0.4404 0.6051 58.6820 45.6213 6,504.0000
prophet              0.4848 0.6365 72.7013 54.6567 6,682.0000
daily_seasonal_naive 0.5305 0.7862 67.3046 49.4682 6,610.0000
sarima               0.5602 0.7758 72.0682 57.4604 6,682.0000

LightGBM vs. baseline: MAE down 0.102 kW (19.2%), RMSE down 24.8%


## Translating forecast accuracy into operational impact

Two different error statistics map to two different operational costs, and
conflating them is a common mistake worth avoiding explicitly:

- **MAE -> imbalance cost.** Day-ahead electricity markets settle the gap
  between a scheduled (forecasted) and actual delivered quantity at an
  imbalance price, typically a premium over the day-ahead clearing price.
  Expected imbalance cost scales with *average absolute* forecast error --
  i.e. MAE, not RMSE.
- **RMSE -> reserve margin.** Grid operators hold spinning/operating
  reserve capacity on standby to cover forecast risk. Reserve sizing is
  conventionally driven by the *spread* of the error distribution --
  RMSE, which penalizes large misses (exactly the failure-case spikes
  Phase 9 found clustering on real anomalous days) more than MAE does.
  LightGBM's larger RMSE improvement (vs. its MAE improvement) is
  specifically a reserve-margin story, not just an average-accuracy one.

Both numbers matter, and they answer different operational questions.

## An illustrative utility-scale cost estimate

**All three assumption values below are the only inputs to this estimate --
change any of them and the result changes accordingly.** This is an
order-of-magnitude illustration of *how* the accuracy improvement above
would translate into avoided imbalance cost at a hypothetical utility
scale, not a validated ROI figure for any specific utility. It also assumes
per-household forecast error carries over unchanged to a pooled portfolio,
which real aggregation effects (errors partially cancel across many
independent households) would likely make conservative -- i.e. this is
probably an upper bound, not an underestimate.

In [4]:
# --- Assumptions (change these, get a different number) ---
PORTFOLIO_HOUSEHOLDS = 10_000       # a mid-sized utility's residential customer count, illustrative
IMBALANCE_PREMIUM_PER_MWH = 20.0    # $/MWh premium of imbalance settlement over day-ahead price, illustrative
HOURS_PER_YEAR = 24 * 365

# --- Mechanical scaling of the real, measured MAE reduction above ---
annual_mae_reduction_kwh_per_household = mae_reduction_kw * HOURS_PER_YEAR
portfolio_annual_mae_reduction_mwh = (
    annual_mae_reduction_kwh_per_household * PORTFOLIO_HOUSEHOLDS / 1000
)
illustrative_annual_savings = portfolio_annual_mae_reduction_mwh * IMBALANCE_PREMIUM_PER_MWH

print(f"MAE reduction per household: {mae_reduction_kw:.3f} kW/hour")
print(f"-> {annual_mae_reduction_kwh_per_household:,.0f} kWh/year of reduced absolute forecast error, per household")
print(f"-> {portfolio_annual_mae_reduction_mwh:,.0f} MWh/year across a {PORTFOLIO_HOUSEHOLDS:,}-household portfolio")
print(f"-> ${illustrative_annual_savings:,.0f}/year in illustrative avoided imbalance cost "
      f"(at ${IMBALANCE_PREMIUM_PER_MWH:.0f}/MWh)")

MAE reduction per household: 0.102 kW/hour
-> 892 kWh/year of reduced absolute forecast error, per household
-> 8,920 MWh/year across a 10,000-household portfolio
-> $178,394/year in illustrative avoided imbalance cost (at $20/MWh)


## Utility demand planning improvements

Beyond the direct imbalance-cost framing above, the specific findings from
Phases 7-9 map onto concrete planning decisions:

- **Lower RMSE -> smaller required reserve margin**, freeing up capacity
  that would otherwise sit on standby for forecast risk -- directly
  actionable given LightGBM's 25% RMSE reduction vs. baseline.
- **Phase 9's failure-case analysis is a targeting tool, not just a
  caveat**: worst-case errors cluster on a small, identifiable set of
  real anomalous days (Phase 3's flagged dates). A production deployment
  could flag forecasts issued for those dates as lower-confidence and
  trigger a larger manual reserve buffer specifically then, rather than
  inflating reserves uniformly every day.
- **Phase 9B's horizon-sensitivity result changes which model should drive
  which planning timescale.** Day-ahead unit commitment (24h) and
  intra-day dispatch (1h) should use LightGBM/LSTM. Weekly generation and
  maintenance scheduling (7d) should lean on LightGBM specifically, not
  SARIMA -- Phase 9B found SARIMA becomes the single worst method of any
  kind at 7d, worse than flat persistence, making it actively harmful for
  week-ahead planning despite being usable at 24h.
- **Seasonal variation matters for planning confidence, not just average
  accuracy.** Phase 9A's per-season breakdown found winter is the hardest
  season for every method (highest MAE across the board) -- reserve
  margins sized on annual-average error would under-cover winter risk and
  over-cover summer, where the naive baseline was nearly competitive with
  the sophisticated models.

## Operational trade-offs between the four approaches

| | Accuracy (24h MAE) | Retraining cost | Horizon flexibility | Interpretability |
|---|---|---|---|---|
| **LightGBM** | Best | Seconds-minutes, CPU only | Native -- direct multi-horizon models retrain cheaply at any horizon (proven: 24 to 168 in Phase 9B with no code changes) | High -- SHAP values (Phase 7) explain individual predictions |
| **LSTM** | Statistically tied with LightGBM (DM p=0.098) | Minutes, GPU/MPS-dependent; prone to overfitting on this dataset size (Phase 8 finding) | Architectural -- a new horizon needs a differently-sized output head and a fresh Optuna search, not a parameter change (why Phase 9B didn't extend it to 7d) | Low -- no direct feature-attribution equivalent used here |
| **Prophet** | Behind both ML models, but 2nd-best specifically at 7d (Phase 9B) | Seconds | Native, but no accuracy edge at short horizons | Very high -- explicit trend/seasonality decomposition, easy to explain to non-technical stakeholders |
| **SARIMA** | Worse than the naive baseline in 3 of 4 seasons (Phase 9A); worst method of any kind at 7d (Phase 9B) | Seconds once order is fixed; auto_arima search itself is slow | Poor -- degrades sharply beyond the horizon its seasonal order (m=24) was designed for | Moderate -- coefficients are interpretable but the seasonal-order choice is a hidden assumption |

The LightGBM-vs-LSTM accuracy gap not being statistically significant
(Phase 9A) matters most here: with no confident accuracy edge for LSTM,
its materially higher retraining cost and lower horizon flexibility are
costs paid for no proven benefit.

## Recommendation

**Primary production model: LightGBM.** Best or statistically-tied-best
accuracy at every horizon tested (1h, 24h, 7d -- Phase 9B), cheapest to
train and retrain, the only model proven to extend cleanly across horizons
without an architecture change, and the only one with a built-in
explainability story (SHAP) for stakeholder trust.

**Secondary use, not a replacement: Prophet for longer-horizon (weekly)
planning conversations specifically**, where its explicit, plain-language
trend/seasonality decomposition is easier to defend to non-technical
stakeholders than LightGBM's feature importances, and where it holds up
reasonably well (2nd-best at 7d).

**Not recommended for production: SARIMA**, given it underperforms the
trivial naive baseline in most seasons and collapses entirely at the
7-day horizon -- keep it only as an interpretability/diagnostic reference,
not a forecast a planning decision should rely on.

**LSTM: a validated research result, not a production pick.** It proved a
real, useful point -- that raw sequence learning with zero manual feature
engineering can match a carefully feature-engineered gradient-boosting
model -- but with no statistically significant accuracy edge and
meaningfully higher operational cost, it isn't the model to deploy given
LightGBM already covers the same ground more cheaply.